# Biochemistry

Split from `01_biochemistry_and_enzyme_kinetics.ipynb` to keep this topic self-contained.

**Navigation:** [Topic overview](./01_biochemistry_and_enzyme_kinetics.ipynb) · [Next: Enzyme Kinetics](./02_enzyme_kinetics.ipynb)


# Biochemistry and Enzyme Kinetics

## Learning Objectives

By the end of this module you will be able to:

- Describe the enzyme-substrate reaction mechanism and extract initial velocities from progress curves
- Derive and fit the Michaelis-Menten equation to obtain Km and Vmax with confidence intervals
- Apply and critically compare linearization methods: Lineweaver-Burk, Eadie-Hofstee, Hanes-Woolf
- Simulate and distinguish competitive, non-competitive, uncompetitive, and mixed inhibition
- Fit the Hill equation and interpret the cooperativity coefficient for allosteric enzymes
- Build stoichiometric matrices for metabolic pathways and understand flux balance analysis

[← Previous: Modern Workflows](../../Tier_3_Applied_Bioinformatics/12_Modern_Workflows/01_modern_workflows.ipynb) | [Next: Genetic Engineering In Silico →](../../Tier_3_Applied_Bioinformatics/14_Genetic_Engineering_In_Silico/01_genetic_engineering_in_silico.ipynb)

---
## 1. Introduction to Enzyme Kinetics

Enzymes are biological catalysts that accelerate chemical reactions without being consumed. They lower the **activation energy** of a reaction by binding substrates in an active site and stabilising the transition state.

The overall catalytic cycle:

```
E + S  ⇌  ES  →  E + P
        k₁      k₂
        k₋₁
```

| Symbol | Meaning |
|--------|------------------------------------------|
| E      | Free enzyme |
| S      | Substrate |
| ES     | Enzyme-substrate (Michaelis) complex |
| P      | Product |
| k₁     | Substrate binding rate |
| k₋₁    | Substrate dissociation rate |
| k₂     | Catalytic rate (kcat) |

**Reaction velocity** (v) is the rate of product formation: v = d[P]/dt.

### 1.1 Spectrophotometric Assays

Most enzyme assays follow absorbance over time. At a fixed wavelength, absorbance A is proportional to the concentration of a chromophoric species (substrate or product).

**Beer-Lambert law:**

$$A = \varepsilon \cdot l \cdot c$$

where:
- A = absorbance (dimensionless)
- ε = molar absorption coefficient (M⁻¹ cm⁻¹)
- l = path length (cm), typically 1 cm
- c = concentration (M)

The **initial velocity** (v₀) is measured from the linear portion of the progress curve (before substrate depletion or product inhibition occurs). It is the slope of [P] vs time at t ≈ 0.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.linalg import null_space
import warnings
warnings.filterwarnings('ignore')

rng = np.random.default_rng(seed=42)

# Beer-Lambert: convert absorbance to concentration
epsilon = 6220   # NADH molar absorption coefficient at 340 nm (M^-1 cm^-1)
path_length = 1  # cm

def absorbance_to_concentration(absorbance, eps=epsilon, l=path_length):
    """Convert absorbance to molar concentration via Beer-Lambert."""
    return absorbance / (eps * l)

print(f"A = 0.622 corresponds to [{absorbance_to_concentration(0.622)*1e6:.1f}] µM NADH")

### 1.2 Extracting Initial Velocity from a Progress Curve

In practice, the linear region of a progress curve spans the first 5–10% of substrate consumption. We fit a straight line to that region.

In [ ]:
# Simulate a product progress curve (molar concentration in µM)
time_s = np.linspace(0, 120, 200)  # seconds

# Simple saturating product accumulation: [P](t) = Pmax * (1 - exp(-t/tau))
P_max_uM = 80.0   # µM total substrate available
tau = 40.0         # time constant (s)
product_uM = P_max_uM * (1 - np.exp(-time_s / tau))
product_uM += rng.normal(0, 0.5, size=len(time_s))  # add measurement noise

# Extract initial velocity: fit linear region (first 10% of Pmax)
linear_mask = product_uM < 0.10 * P_max_uM
if linear_mask.sum() < 3:
    linear_mask = np.arange(len(time_s)) < 15

slope, intercept = np.polyfit(time_s[linear_mask], product_uM[linear_mask], 1)
v0_uM_per_s = slope  # µM/s

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(time_s, product_uM, color='steelblue', lw=1.5, label='Progress curve')
ax.plot(time_s[linear_mask],
        slope * time_s[linear_mask] + intercept,
        color='tomato', lw=2, linestyle='--', label=f'Linear fit (v₀ = {v0_uM_per_s:.2f} µM/s)')
ax.set_xlabel('Time (s)')
ax.set_ylabel('[Product] (µM)')
ax.set_title('Product Progress Curve and Initial Velocity')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Estimated initial velocity v₀ = {v0_uM_per_s:.3f} µM/s")

---
## 2. Michaelis-Menten Kinetics

Assuming **quasi-steady-state** for [ES] (d[ES]/dt ≈ 0), the Michaelis-Menten equation describes reaction velocity as a function of substrate concentration:

$$v = \frac{V_{\max} \cdot [S]}{K_m + [S]}$$

| Parameter | Meaning |
|-----------|------------------------------------------------------------|
| Vmax      | Maximum velocity (all enzyme sites saturated) |
| Km        | Michaelis constant = [S] at which v = Vmax/2; reflects affinity |
| kcat      | Turnover number = Vmax / [E]total |

- **Low Km** → high affinity (enzyme is half-saturated at low [S])
- **High Km** → low affinity

At low [S]: v ≈ (Vmax/Km)·[S]  — first-order  
At high [S]: v ≈ Vmax           — zero-order (saturated)

In [ ]:
def michaelis_menten(substrate_conc, v_max, k_m):
    """Michaelis-Menten velocity."""
    return (v_max * substrate_conc) / (k_m + substrate_conc)

# True kinetic parameters
true_vmax = 100.0  # µM/s
true_km   = 5.0    # µM

# Substrate concentrations spanning 0.1*Km to 10*Km (good experimental design)
substrate_uM = np.array([0.5, 1.0, 2.0, 3.0, 5.0, 7.5, 10.0, 15.0, 20.0, 30.0, 50.0])

# Generate synthetic velocities with 5% Gaussian noise
true_velocities = michaelis_menten(substrate_uM, true_vmax, true_km)
noise_sd = 0.05 * true_velocities
observed_velocities = true_velocities + rng.normal(0, noise_sd)
observed_velocities = np.clip(observed_velocities, 0, None)

print("Substrate [µM] | Observed velocity [µM/s]")
print("-" * 40)
for s, v in zip(substrate_uM, observed_velocities):
    print(f"  {s:6.1f}       |  {v:8.3f}")

### 2.1 Fitting Km and Vmax with `scipy.optimize.curve_fit`

`curve_fit` uses nonlinear least squares (Levenberg-Marquardt). It returns the best-fit parameters and the covariance matrix, from which we compute standard errors and 95% confidence intervals.

In [ ]:
# Fit Michaelis-Menten model
p0 = [80.0, 3.0]  # initial guesses: [vmax, km]
bounds = ([0, 0], [np.inf, np.inf])

popt, pcov = curve_fit(
    michaelis_menten,
    substrate_uM,
    observed_velocities,
    p0=p0,
    bounds=bounds,
    maxfev=5000
)

fitted_vmax, fitted_km = popt
perr = np.sqrt(np.diag(pcov))  # standard errors

# 95% CI: parameter ± 1.96 * SE
ci_vmax = 1.96 * perr[0]
ci_km   = 1.96 * perr[1]

print(f"Fitted Vmax = {fitted_vmax:.2f} ± {ci_vmax:.2f} µM/s  (true: {true_vmax})")
print(f"Fitted Km   = {fitted_km:.2f}  ± {ci_km:.2f}  µM    (true: {true_km})")

In [ ]:
# Plot Michaelis-Menten curve with data
s_smooth = np.linspace(0, 55, 300)
v_fit    = michaelis_menten(s_smooth, fitted_vmax, fitted_km)

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(substrate_uM, observed_velocities,
           color='steelblue', zorder=5, s=60, label='Observed data')
ax.plot(s_smooth, v_fit, color='tomato', lw=2,
        label=f'MM fit: Vmax={fitted_vmax:.1f}, Km={fitted_km:.2f} µM')
ax.axhline(fitted_vmax, color='gray', linestyle=':', lw=1)
ax.axhline(fitted_vmax / 2, color='gray', linestyle=':', lw=1)
ax.axvline(fitted_km, color='gray', linestyle=':', lw=1)
ax.annotate('Vmax', xy=(55, fitted_vmax), va='bottom', ha='right', color='gray')
ax.annotate('Vmax/2', xy=(55, fitted_vmax / 2), va='bottom', ha='right', color='gray')
ax.annotate('Km', xy=(fitted_km, 2), va='bottom', ha='left', color='gray')
ax.set_xlabel('[S] (µM)')
ax.set_ylabel('v (µM/s)')
ax.set_title('Michaelis-Menten Kinetics')
ax.legend()
plt.tight_layout()
plt.show()

### 2.2 Confidence Intervals via Parametric Bootstrap

The covariance matrix from `curve_fit` assumes Gaussian errors and linearity near the optimum. A parametric bootstrap gives a distribution over the fitted curve, making confidence bands visible.

In [ ]:
# Parametric bootstrap: resample from fitted model + residual noise
n_bootstrap = 500
residuals = observed_velocities - michaelis_menten(substrate_uM, fitted_vmax, fitted_km)
residual_sd = np.std(residuals)

bootstrap_curves = np.zeros((n_bootstrap, len(s_smooth)))
bootstrap_params = np.zeros((n_bootstrap, 2))

for i in range(n_bootstrap):
    v_boot = michaelis_menten(substrate_uM, fitted_vmax, fitted_km) + rng.normal(0, residual_sd, size=len(substrate_uM))
    v_boot = np.clip(v_boot, 0, None)
    try:
        p_boot, _ = curve_fit(michaelis_menten, substrate_uM, v_boot, p0=popt, bounds=bounds, maxfev=2000)
        bootstrap_curves[i] = michaelis_menten(s_smooth, *p_boot)
        bootstrap_params[i] = p_boot
    except RuntimeError:
        bootstrap_curves[i] = np.nan

ci_low  = np.nanpercentile(bootstrap_curves, 2.5, axis=0)
ci_high = np.nanpercentile(bootstrap_curves, 97.5, axis=0)

fig, ax = plt.subplots(figsize=(7, 4))
ax.fill_between(s_smooth, ci_low, ci_high, alpha=0.25, color='tomato', label='95% CI (bootstrap)')
ax.plot(s_smooth, v_fit, color='tomato', lw=2, label='Best fit')
ax.scatter(substrate_uM, observed_velocities, color='steelblue', zorder=5, s=60, label='Data')
ax.set_xlabel('[S] (µM)')
ax.set_ylabel('v (µM/s)')
ax.set_title('Michaelis-Menten Fit with 95% Confidence Band')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Bootstrap Vmax: {np.nanmean(bootstrap_params[:,0]):.2f} ± {np.nanstd(bootstrap_params[:,0]):.2f} µM/s")
print(f"Bootstrap Km:   {np.nanmean(bootstrap_params[:,1]):.2f} ± {np.nanstd(bootstrap_params[:,1]):.2f} µM")

---
## 3. Linearization Methods

Before nonlinear regression was widely available, researchers linearised the Michaelis-Menten equation. Each transformation distorts errors differently.

### 3.1 Lineweaver-Burk (Double Reciprocal)

Take reciprocals of both sides of MM:

$$\frac{1}{v} = \frac{K_m}{V_{\max}} \cdot \frac{1}{[S]} + \frac{1}{V_{\max}}$$

Plot 1/v vs 1/[S]: slope = Km/Vmax, y-intercept = 1/Vmax, x-intercept = -1/Km.

**Problem:** Small velocities (low [S]) are highly amplified in reciprocal space, severely distorting error structure.

### 3.2 Eadie-Hofstee

$$v = V_{\max} - K_m \cdot \frac{v}{[S]}$$

Plot v vs v/[S]: slope = -Km, y-intercept = Vmax.

### 3.3 Hanes-Woolf

$$\frac{[S]}{v} = \frac{K_m}{V_{\max}} + \frac{1}{V_{\max}} \cdot [S]$$

Plot [S]/v vs [S]: slope = 1/Vmax, y-intercept = Km/Vmax. **Most uniform error distribution** of the three.

In [ ]:
# Compute all three linearizations
inv_s = 1.0 / substrate_uM
inv_v = 1.0 / observed_velocities
v_over_s = observed_velocities / substrate_uM
s_over_v = substrate_uM / observed_velocities

# Lineweaver-Burk fit
lb_slope, lb_intercept = np.polyfit(inv_s, inv_v, 1)
lb_vmax = 1.0 / lb_intercept
lb_km   = lb_slope * lb_vmax

# Eadie-Hofstee fit
eh_slope, eh_intercept = np.polyfit(v_over_s, observed_velocities, 1)
eh_km   = -eh_slope
eh_vmax = eh_intercept

# Hanes-Woolf fit
hw_slope, hw_intercept = np.polyfit(substrate_uM, s_over_v, 1)
hw_vmax = 1.0 / hw_slope
hw_km   = hw_intercept * hw_vmax

print(f"{'Method':<20} {'Vmax (µM/s)':>12} {'Km (µM)':>10}")
print("-" * 45)
print(f"{'True values':<20} {true_vmax:>12.2f} {true_km:>10.2f}")
print(f"{'Nonlinear (NLS)':<20} {fitted_vmax:>12.2f} {fitted_km:>10.2f}")
print(f"{'Lineweaver-Burk':<20} {lb_vmax:>12.2f} {lb_km:>10.2f}")
print(f"{'Eadie-Hofstee':<20} {eh_vmax:>12.2f} {eh_km:>10.2f}")
print(f"{'Hanes-Woolf':<20} {hw_vmax:>12.2f} {hw_km:>10.2f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# --- Lineweaver-Burk ---
ax = axes[0]
x_lb = np.linspace(min(inv_s) * 0.8, max(inv_s) * 1.1, 100)
ax.scatter(inv_s, inv_v, color='steelblue', zorder=5, s=50)
ax.plot(x_lb, lb_slope * x_lb + lb_intercept, color='tomato', lw=2)
ax.axvline(0, color='k', lw=0.5)
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('1/[S] (µM⁻¹)')
ax.set_ylabel('1/v (s/µM)')
ax.set_title('Lineweaver-Burk')

# --- Eadie-Hofstee ---
ax = axes[1]
x_eh = np.linspace(min(v_over_s) * 0.8, max(v_over_s) * 1.1, 100)
ax.scatter(v_over_s, observed_velocities, color='steelblue', zorder=5, s=50)
ax.plot(x_eh, eh_slope * x_eh + eh_intercept, color='tomato', lw=2)
ax.set_xlabel('v/[S] (s⁻¹)')
ax.set_ylabel('v (µM/s)')
ax.set_title('Eadie-Hofstee')

# --- Hanes-Woolf ---
ax = axes[2]
x_hw = np.linspace(min(substrate_uM) * 0.8, max(substrate_uM) * 1.1, 100)
ax.scatter(substrate_uM, s_over_v, color='steelblue', zorder=5, s=50)
ax.plot(x_hw, hw_slope * x_hw + hw_intercept, color='tomato', lw=2)
ax.set_xlabel('[S] (µM)')
ax.set_ylabel('[S]/v (s)')
ax.set_title('Hanes-Woolf')

plt.suptitle('Linearization Methods Compared', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### 3.4 Why Lineweaver-Burk Distorts Errors

If the original velocity measurements have constant percentage error (as is typical), then:

- In the original space, small [S] gives small v with small absolute error
- Taking 1/v **amplifies** the relative error at small [S] (which map to large 1/[S])
- The rightmost points on the LB plot (which have the largest leverage) are the most uncertain
- This leads to **systematically biased** Km and Vmax estimates

**Hanes-Woolf** applies 1/v as a *divisor* to [S], spreading error more evenly.

**Modern recommendation:** Use nonlinear least squares (NLS) directly on the MM equation — it respects the original error structure and requires no transformation.

In [ ]:
# Visualize error amplification in Lineweaver-Burk
noise_frac = 0.05  # 5% relative error on v

# For each [S], propagate: sigma(1/v) ≈ sigma(v) / v^2 = noise_frac / v
true_v = michaelis_menten(substrate_uM, true_vmax, true_km)
sigma_inv_v = noise_frac / true_v   # error on 1/v

fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(inv_s, 1/true_v, yerr=sigma_inv_v,
            fmt='o', color='steelblue', capsize=4, label='1/v ± propagated σ')
ax.set_xlabel('1/[S] (µM⁻¹)')
ax.set_ylabel('1/v (s/µM)')
ax.set_title('Error Amplification in Lineweaver-Burk Space')
ax.legend()
plt.tight_layout()
plt.show()

print("Note: Error bars grow dramatically at high 1/[S] (low substrate)")
print("These points dominate the linear regression fit — biasing Km and Vmax.")

---
## 4. Enzyme Inhibition

Inhibitors reduce enzyme activity. The pattern of inhibition — how it affects Km and Vmax — reveals where the inhibitor binds.

| Inhibition Type | Effect on Km | Effect on Vmax | Inhibitor binds |
|-----------------|-------------|---------------|------------------|
| Competitive     | ↑ (apparent) | unchanged     | Free enzyme (E) |
| Non-competitive | unchanged    | ↓             | E and ES |
| Uncompetitive   | ↓           | ↓             | ES complex only |
| Mixed           | ↑ or ↓      | ↓             | E (α) and ES (α') |

### Equations

Let [I] = inhibitor concentration, Ki = inhibition constant.

$$\alpha = 1 + \frac{[I]}{K_i}, \quad \alpha' = 1 + \frac{[I]}{K_i'}$$

| Type | Equation |
|------|----------|
| Competitive | v = Vmax·[S] / (α·Km + [S]) |
| Non-competitive | v = (Vmax/α)·[S] / (Km + [S]) |
| Uncompetitive | v = (Vmax/α')·[S] / (Km/α' + [S]) |
| Mixed | v = Vmax·[S] / (α·Km + α'·[S]) |

In [ ]:
def competitive_inhibition(substrate_conc, v_max, k_m, inhibitor_conc, k_i):
    alpha = 1 + inhibitor_conc / k_i
    return (v_max * substrate_conc) / (alpha * k_m + substrate_conc)

def noncompetitive_inhibition(substrate_conc, v_max, k_m, inhibitor_conc, k_i):
    alpha = 1 + inhibitor_conc / k_i
    return (v_max / alpha * substrate_conc) / (k_m + substrate_conc)

def uncompetitive_inhibition(substrate_conc, v_max, k_m, inhibitor_conc, k_i_prime):
    alpha_prime = 1 + inhibitor_conc / k_i_prime
    return (v_max / alpha_prime * substrate_conc) / (k_m / alpha_prime + substrate_conc)

def mixed_inhibition(substrate_conc, v_max, k_m, inhibitor_conc, k_i, k_i_prime):
    alpha       = 1 + inhibitor_conc / k_i
    alpha_prime = 1 + inhibitor_conc / k_i_prime
    return (v_max * substrate_conc) / (alpha * k_m + alpha_prime * substrate_conc)

# Parameters
vmax_inh = 100.0  # µM/s
km_inh   = 5.0    # µM
ki       = 2.0    # µM
ki_prime = 4.0    # µM (for mixed inhibition)

inhibitor_concs = [0.0, 2.0, 5.0]  # µM [I]
s_range = np.linspace(0.1, 50, 200)

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
titles = ['Competitive', 'Non-Competitive', 'Uncompetitive', 'Mixed']
colors = ['steelblue', 'darkorange', 'tomato']

for ax, title in zip(axes.flat, titles):
    for inh_c, col in zip(inhibitor_concs, colors):
        if title == 'Competitive':
            v = competitive_inhibition(s_range, vmax_inh, km_inh, inh_c, ki)
        elif title == 'Non-Competitive':
            v = noncompetitive_inhibition(s_range, vmax_inh, km_inh, inh_c, ki)
        elif title == 'Uncompetitive':
            v = uncompetitive_inhibition(s_range, vmax_inh, km_inh, inh_c, ki_prime)
        else:
            v = mixed_inhibition(s_range, vmax_inh, km_inh, inh_c, ki, ki_prime)
        label = f'[I] = {inh_c:.0f} µM'
        ax.plot(s_range, v, color=col, lw=2, label=label)
    ax.set_xlabel('[S] (µM)')
    ax.set_ylabel('v (µM/s)')
    ax.set_title(title)
    ax.legend(fontsize=9)

plt.suptitle('Enzyme Inhibition Patterns (Vmax=100, Km=5, Ki=2 µM)', fontsize=13)
plt.tight_layout()
plt.show()

### 4.1 Generating Inhibition Data and Fitting Models

To determine inhibition type experimentally, measure v at multiple [S] and several [I] values. Then fit each inhibition model and compare quality of fit.

In [ ]:
# Generate synthetic data for competitive inhibition
substrate_grid = np.array([1.0, 2.0, 5.0, 10.0, 20.0, 50.0])  # µM
inhibitor_grid = np.array([0.0, 1.0, 3.0])  # µM

true_ki_comp = 3.0  # µM

inh_data = {}  # {[I]: array of velocities}
for inh_c in inhibitor_grid:
    v_true = competitive_inhibition(substrate_grid, vmax_inh, km_inh, inh_c, true_ki_comp)
    noise  = rng.normal(0, 0.03 * v_true)
    inh_data[inh_c] = np.clip(v_true + noise, 0, None)

# Fit competitive model for each [I] to extract apparent Km
print("Fitting MM to each [I] concentration separately:")
print(f"{'[I] (µM)':>10} | {'Vmax_app':>10} | {'Km_app':>10}")
print("-" * 38)

apparent_kms  = []
apparent_vmaxs = []

for inh_c in inhibitor_grid:
    v_obs = inh_data[inh_c]
    p_inh, _ = curve_fit(michaelis_menten, substrate_grid, v_obs,
                          p0=[90, 5], bounds=([0, 0], [np.inf, np.inf]), maxfev=3000)
    apparent_kms.append(p_inh[1])
    apparent_vmaxs.append(p_inh[0])
    print(f"{inh_c:>10.1f} | {p_inh[0]:>10.2f} | {p_inh[1]:>10.2f}")

print()
print("Competitive inhibition signature: Km_app increases with [I], Vmax stays ~constant.")

In [ ]:
# Determining Ki from apparent Km values
# For competitive inhibition: Km_app = Km * (1 + [I]/Ki)
# Linear regression: Km_app vs [I] -> slope = Km/Ki, intercept = Km

apparent_kms_arr = np.array(apparent_kms)

slope_ki, intercept_ki = np.polyfit(inhibitor_grid, apparent_kms_arr, 1)
estimated_km_from_intercept = intercept_ki
estimated_ki = intercept_ki / slope_ki

print(f"Estimated Km (intercept) = {estimated_km_from_intercept:.2f} µM  (true: {km_inh})")
print(f"Estimated Ki             = {estimated_ki:.2f} µM  (true: {true_ki_comp})")

fig, ax = plt.subplots(figsize=(6, 4))
x_plot = np.linspace(-estimated_ki, max(inhibitor_grid) * 1.1, 100)
ax.scatter(inhibitor_grid, apparent_kms_arr, color='steelblue', s=80, zorder=5)
ax.plot(x_plot, slope_ki * x_plot + intercept_ki, color='tomato', lw=2)
ax.axvline(-estimated_ki, color='gray', linestyle='--', lw=1)
ax.annotate(f'-Ki = {-estimated_ki:.1f} µM', xy=(-estimated_ki, 0.5),
            xytext=(-estimated_ki + 0.5, 2), fontsize=10, color='gray')
ax.set_xlabel('[I] (µM)')
ax.set_ylabel('Apparent Km (µM)')
ax.set_title('Dixon Plot for Ki Estimation')
plt.tight_layout()
plt.show()

### 4.2 Lineweaver-Burk Patterns for Each Inhibition Type

The diagnostic pattern in a Lineweaver-Burk plot identifies inhibition type:

| Type | LB pattern |
|------|------------|
| Competitive | Lines intersect on y-axis (same 1/Vmax) |
| Non-competitive | Lines intersect on x-axis (same -1/Km) |
| Uncompetitive | Parallel lines |
| Mixed | Lines intersect in second quadrant |

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
inh_concs_lb = [0.0, 2.0, 5.0]
colors_lb    = ['steelblue', 'darkorange', 'tomato']
s_lb = np.array([0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 50.0])
inv_s_lb = 1 / s_lb
x_line = np.linspace(-0.3, max(inv_s_lb) * 1.1, 200)

inhibition_funcs = {
    'Competitive':      lambda s, inh: competitive_inhibition(s, vmax_inh, km_inh, inh, ki),
    'Non-Competitive':  lambda s, inh: noncompetitive_inhibition(s, vmax_inh, km_inh, inh, ki),
    'Uncompetitive':    lambda s, inh: uncompetitive_inhibition(s, vmax_inh, km_inh, inh, ki_prime),
    'Mixed':            lambda s, inh: mixed_inhibition(s, vmax_inh, km_inh, inh, ki, ki_prime),
}

for ax, (title, fn) in zip(axes, inhibition_funcs.items()):
    for inh_c, col in zip(inh_concs_lb, colors_lb):
        v_vals = fn(s_lb, inh_c)
        slope_l, intercept_l = np.polyfit(inv_s_lb, 1/v_vals, 1)
        ax.scatter(inv_s_lb, 1/v_vals, color=col, s=30, zorder=5)
        ax.plot(x_line, slope_l * x_line + intercept_l,
                color=col, lw=1.5, label=f'[I]={inh_c:.0f}')
    ax.axvline(0, color='k', lw=0.5)
    ax.axhline(0, color='k', lw=0.5)
    ax.set_xlabel('1/[S]')
    ax.set_ylabel('1/v')
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.set_xlim(-0.3, max(inv_s_lb) * 1.1)

plt.suptitle('Lineweaver-Burk Patterns for Each Inhibition Type', fontsize=13)
plt.tight_layout()
plt.show()

### 4.3 Model Comparison: Identifying Inhibition Type

Given data, fit each inhibition model and use **Akaike Information Criterion (AIC)** to select the most parsimonious model:

$$AIC = 2k - 2\ln(\hat{L})$$

where k = number of parameters, and L̂ is the maximised likelihood. Lower AIC = better model (penalised for complexity).

In [ ]:
# Generate data under competitive inhibition at one [I] value
test_inhibitor_conc = 3.0  # µM
v_test = competitive_inhibition(substrate_uM, vmax_inh, km_inh, test_inhibitor_conc, true_ki_comp)
v_test_obs = v_test + rng.normal(0, 0.04 * v_test)

def aic(n_params, residuals):
    n = len(residuals)
    rss = np.sum(residuals**2)
    # MLE estimate of sigma^2
    sigma2 = rss / n
    log_lik = -n/2 * np.log(2 * np.pi * sigma2) - rss / (2 * sigma2)
    return 2 * n_params - 2 * log_lik

models = {
    'Competitive':     (lambda s, vm, km, ki_: competitive_inhibition(s, vm, km, test_inhibitor_conc, ki_),
                        [90, 4, 2], 3),
    'Non-Competitive': (lambda s, vm, km, ki_: noncompetitive_inhibition(s, vm, km, test_inhibitor_conc, ki_),
                        [90, 4, 2], 3),
    'Uncompetitive':   (lambda s, vm, km, kip: uncompetitive_inhibition(s, vm, km, test_inhibitor_conc, kip),
                        [90, 4, 2], 3),
}

print(f"{'Model':<18} {'AIC':>10} {'Vmax_fit':>10} {'Km_fit':>10} {'Ki_fit':>10}")
print("-" * 58)
for model_name, (fn, p0_m, k_params) in models.items():
    try:
        pm, _ = curve_fit(fn, substrate_uM, v_test_obs, p0=p0_m,
                          bounds=([0,0,0.01],[np.inf,np.inf,np.inf]), maxfev=5000)
        resid = v_test_obs - fn(substrate_uM, *pm)
        aic_val = aic(k_params, resid)
        print(f"{model_name:<18} {aic_val:>10.2f} {pm[0]:>10.2f} {pm[1]:>10.2f} {pm[2]:>10.2f}")
    except RuntimeError:
        print(f"{model_name:<18} {'FAILED':>10}")

print()
print("Lowest AIC identifies the most likely inhibition type.")

---
## 5. Allosteric Enzymes and the Hill Equation

Some enzymes show **sigmoidal** (S-shaped) kinetics rather than hyperbolic. This occurs in **allosteric enzymes** with multiple subunits that exhibit **cooperativity**: binding of one substrate molecule changes the affinity of remaining sites.

### Hill Equation

$$v = \frac{V_{\max} \cdot [S]^n}{K_{0.5}^n + [S]^n}$$

| Parameter | Meaning |
|-----------|--------------------------------------------|
| n (Hill coefficient) | Cooperativity: n=1 hyperbolic, n>1 positive cooperativity, n<1 negative |
| K₀.₅ | [S] at half-maximal velocity |

**Biological example:** Hemoglobin binds O₂ with n ≈ 2.7–3.0. The sigmoidal curve allows efficient loading in the lungs ([O₂] high) and efficient unloading in tissues ([O₂] low).

In [ ]:
def hill_equation(substrate_conc, v_max, k_half, hill_n):
    """Hill equation for cooperative substrate binding."""
    return (v_max * substrate_conc**hill_n) / (k_half**hill_n + substrate_conc**hill_n)

# Compare MM (n=1) vs various Hill coefficients
s_hill = np.linspace(0, 20, 300)
k_half_val = 5.0
vmax_hill  = 100.0

fig, ax = plt.subplots(figsize=(7, 4))
for n_val, col in zip([0.5, 1.0, 2.0, 3.0], ['purple', 'steelblue', 'darkorange', 'tomato']):
    v_hill = hill_equation(s_hill, vmax_hill, k_half_val, n_val)
    ax.plot(s_hill, v_hill, color=col, lw=2, label=f'n = {n_val}')

ax.axhline(vmax_hill / 2, color='gray', linestyle=':', lw=1)
ax.axvline(k_half_val, color='gray', linestyle=':', lw=1)
ax.set_xlabel('[S] (µM)')
ax.set_ylabel('v (µM/s)')
ax.set_title('Hill Equation: Effect of Cooperativity Coefficient (n)')
ax.legend(title='Hill n')
plt.tight_layout()
plt.show()

### 5.1 Hemoglobin O₂ Binding Example

The fractional saturation of hemoglobin (Y) with O₂ as a function of partial pressure:

$$Y = \frac{pO_2^n}{P_{50}^n + pO_2^n}$$

where P₅₀ is the pO₂ at 50% saturation (~26 mmHg in human blood, n ≈ 2.8).

In [ ]:
# Hemoglobin oxygen binding
true_p50   = 26.0  # mmHg
true_n_hb  = 2.8   # Hill coefficient for hemoglobin

po2_mmhg = np.array([2, 5, 10, 15, 20, 26, 35, 45, 60, 80, 100])

# Generate fractional saturation with noise
y_true = hill_equation(po2_mmhg, 1.0, true_p50, true_n_hb)
y_obs  = y_true + rng.normal(0, 0.015)
y_obs  = np.clip(y_obs, 0, 1)

# Fit Hill equation to hemoglobin data
popt_hb, pcov_hb = curve_fit(
    hill_equation, po2_mmhg, y_obs,
    p0=[1.0, 25.0, 2.5],
    bounds=([0.8, 5, 0.5], [1.2, 100, 6]),
    maxfev=5000
)

perr_hb = np.sqrt(np.diag(pcov_hb))
print(f"Fitted Ymax = {popt_hb[0]:.3f} ± {perr_hb[0]:.3f}")
print(f"Fitted P50  = {popt_hb[1]:.2f} ± {perr_hb[1]:.2f} mmHg  (true: {true_p50})")
print(f"Fitted n    = {popt_hb[2]:.2f} ± {perr_hb[2]:.2f}        (true: {true_n_hb})")

po2_smooth = np.linspace(0, 110, 300)
y_fit_hb = hill_equation(po2_smooth, *popt_hb)
y_mm_hb  = hill_equation(po2_smooth, 1.0, true_p50, 1.0)  # Hypothetical non-cooperative

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(po2_mmhg, y_obs, color='tomato', zorder=5, s=60, label='Observed')
ax.plot(po2_smooth, y_fit_hb, color='tomato', lw=2,
        label=f'Hill fit (n={popt_hb[2]:.2f}, P50={popt_hb[1]:.1f})')
ax.plot(po2_smooth, y_mm_hb, color='steelblue', lw=2, linestyle='--',
        label='Non-cooperative (n=1)')
ax.axvline(26, color='gray', linestyle=':', lw=1)
ax.axvline(100, color='lightblue', linestyle='-', lw=1, alpha=0.5)
ax.annotate('Lung pO₂', xy=(100, 0.05), ha='right', color='steelblue', fontsize=9)
ax.axvline(40, color='lightcoral', linestyle='-', lw=1, alpha=0.5)
ax.annotate('Tissue pO₂', xy=(40, 0.05), ha='left', color='tomato', fontsize=9)
ax.set_xlabel('pO₂ (mmHg)')
ax.set_ylabel('Fractional O₂ Saturation (Y)')
ax.set_title('Hemoglobin Oxygen Binding Curve')
ax.legend()
plt.tight_layout()
plt.show()

### 5.2 The Hill Plot

The **Hill plot** linearises the Hill equation:

$$\log\left(\frac{v}{V_{\max} - v}\right) = n \cdot \log([S]) - n \cdot \log(K_{0.5})$$

Plot log(v/(Vmax-v)) vs log([S]). The slope = n (Hill coefficient), y-intercept = -n·log(K₀.₅).

At low and high [S], the slope approaches 1 (reflecting individual binding events). The maximum slope at mid-range saturation gives the Hill coefficient.

In [ ]:
# Hill plot for hemoglobin data
# Exclude saturation extremes (Y near 0 or 1 give log(inf) or log(0))
mask_hill = (y_obs > 0.05) & (y_obs < 0.95)
log_ratio  = np.log10(y_obs[mask_hill] / (1 - y_obs[mask_hill]))
log_po2    = np.log10(po2_mmhg[mask_hill])

slope_hill, intercept_hill = np.polyfit(log_po2, log_ratio, 1)
n_from_hill_plot = slope_hill
k_half_from_plot = 10**(-intercept_hill / slope_hill)

x_hill_line = np.linspace(min(log_po2) * 0.95, max(log_po2) * 1.05, 100)

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(log_po2, log_ratio, color='tomato', s=60, zorder=5)
ax.plot(x_hill_line, slope_hill * x_hill_line + intercept_hill, color='steelblue', lw=2)
ax.axhline(0, color='gray', linestyle=':', lw=1)
ax.set_xlabel('log₁₀(pO₂)')
ax.set_ylabel('log₁₀(Y / (1 − Y))')
ax.set_title(f'Hill Plot: slope (n) = {n_from_hill_plot:.2f},  K₀.₅ = {k_half_from_plot:.1f} mmHg')
plt.tight_layout()
plt.show()

print(f"Hill coefficient from plot: n = {n_from_hill_plot:.2f}  (true: {true_n_hb})")
print(f"K₀.₅ from plot: {k_half_from_plot:.2f} mmHg  (true: {true_p50})")

---
## 6. Enzyme Classification

The **Enzyme Commission (EC) system** classifies enzymes by the reaction they catalyse. Every enzyme has a four-number code: EC n1.n2.n3.n4.

| EC Class | Name | Reaction Type | Example |
|----------|------|---------------|---------|
| EC 1 | Oxidoreductases | Oxidation/reduction | Lactate dehydrogenase |
| EC 2 | Transferases | Transfer of functional groups | Hexokinase |
| EC 3 | Hydrolases | Hydrolysis | Trypsin |
| EC 4 | Lyases | Addition/removal without water | Pyruvate decarboxylase |
| EC 5 | Isomerases | Intramolecular rearrangements | Phosphoglucose isomerase |
| EC 6 | Ligases | Bond formation using ATP | Aminoacyl-tRNA synthetase |

**Enzyme specificity** ranges from absolute (one substrate only) to broad (many similar substrates). **Promiscuous** enzymes catalyse multiple, chemically distinct reactions — this is thought to be a source of metabolic innovation in evolution.

In [ ]:
# Conceptual enzyme database query (using a local dictionary as a stand-in)
enzyme_database = {
    'EC 1.1.1.27': {'name': 'L-lactate dehydrogenase', 'reaction': 'L-lactate + NAD+ ⇌ pyruvate + NADH',
                    'class': 'Oxidoreductase', 'organism': 'Homo sapiens'},
    'EC 2.7.1.1':  {'name': 'Hexokinase', 'reaction': 'ATP + D-hexose → ADP + D-hexose-6-phosphate',
                    'class': 'Transferase', 'organism': 'Homo sapiens'},
    'EC 3.4.21.4':  {'name': 'Trypsin', 'reaction': 'Cleaves Arg/Lys peptide bonds',
                     'class': 'Hydrolase', 'organism': 'Bos taurus'},
    'EC 4.1.1.1':  {'name': 'Pyruvate decarboxylase', 'reaction': 'Pyruvate → acetaldehyde + CO2',
                    'class': 'Lyase', 'organism': 'Saccharomyces cerevisiae'},
    'EC 5.3.1.9':  {'name': 'Phosphoglucose isomerase', 'reaction': 'D-glucose 6-P ⇌ D-fructose 6-P',
                    'class': 'Isomerase', 'organism': 'Homo sapiens'},
    'EC 6.1.1.1':  {'name': 'Tyrosine-tRNA ligase', 'reaction': 'ATP + L-Tyr + tRNA(Tyr) → AMP + PPi + L-Tyr-tRNA',
                    'class': 'Ligase', 'organism': 'Homo sapiens'},
}

def query_enzyme(ec_number):
    entry = enzyme_database.get(ec_number)
    if entry is None:
        return f"EC number {ec_number} not found."
    return (f"{ec_number}: {entry['name']}\n"
            f"  Class:    {entry['class']}\n"
            f"  Reaction: {entry['reaction']}\n"
            f"  Organism: {entry['organism']}")

for ec in ['EC 1.1.1.27', 'EC 2.7.1.1', 'EC 5.3.1.9']:
    print(query_enzyme(ec))
    print()

In [ ]:
# Distribution of EC classes in our mini-database
ec_classes = [e['class'] for e in enzyme_database.values()]
class_counts = {c: ec_classes.count(c) for c in set(ec_classes)}

fig, ax = plt.subplots(figsize=(6, 3))
ax.barh(list(class_counts.keys()), list(class_counts.values()), color='steelblue')
ax.set_xlabel('Count in Example Database')
ax.set_title('Enzyme Classification by EC Class')
plt.tight_layout()
plt.show()

---
## 7. Metabolic Pathways

A **metabolic pathway** is a series of enzyme-catalysed reactions converting metabolites. The stoichiometry of all reactions is captured in the **stoichiometric matrix S**.

### Stoichiometric Matrix

- Rows = metabolites (m)
- Columns = reactions (n)
- S[i,j] = stoichiometric coefficient of metabolite i in reaction j (negative = consumed, positive = produced)

At **steady state**: S·v = 0, where v is the flux vector.

The null space of S gives all steady-state flux distributions.

In [ ]:
# Glycolysis fragment: glucose → fructose-6-P → fructose-1,6-bisP → ... → pyruvate
# Simplified 5-reaction, 4-metabolite pathway

# Metabolites: Glucose (glc), Glucose-6-P (g6p), Fructose-6-P (f6p), Fructose-1,6-BP (fbp)
# Reactions:
#   R1: Glc -> G6P  (Hexokinase)
#   R2: G6P -> F6P  (Phosphoglucose isomerase)
#   R3: F6P -> FBP  (Phosphofructokinase)
#   R4: FBP -> ...  (Aldolase; FBP leaves system)
#   R5: G6P -> ...  (Glucose-6-phosphatase; G6P leaves system)

metabolites = ['Glucose', 'G6P', 'F6P', 'FBP']
reactions   = ['Hexokinase', 'PGI', 'PFK', 'Aldolase', 'G6Pase']

#               HK    PGI   PFK   Aldo  G6Pase
S_matrix = np.array([
    [-1,    0,    0,    0,    0  ],   # Glucose
    [ 1,   -1,    0,    0,   -1  ],   # G6P
    [ 0,    1,   -1,    0,    0  ],   # F6P
    [ 0,    0,    1,   -1,    0  ],   # FBP
], dtype=float)

print("Stoichiometric matrix S (rows=metabolites, cols=reactions):")
print(f"{'':12}", end='')
for r in reactions:
    print(f"{r:>10}", end='')
print()
for i, met in enumerate(metabolites):
    print(f"{met:<12}", end='')
    for j in range(len(reactions)):
        print(f"{int(S_matrix[i,j]):>10}", end='')
    print()
print(f"\nMatrix shape: {S_matrix.shape} ({len(metabolites)} metabolites × {len(reactions)} reactions)")

### 7.1 Flux Balance Analysis Concepts

**Flux Balance Analysis (FBA)** finds steady-state flux distributions (S·v = 0) that optimise a biological objective (e.g., maximise ATP production) subject to bounds on each flux.

The **null space** of S is the set of all v such that S·v = 0. Each basis vector of the null space is an **elementary flux mode** — a minimal pathway that can operate at steady state.

In [ ]:
# Compute null space of S (steady-state flux modes)
ns = null_space(S_matrix)

print(f"Null space dimension: {ns.shape[1]} (degrees of freedom in flux distribution)")
print()

for i in range(ns.shape[1]):
    flux_mode = ns[:, i]
    # Normalize to largest absolute flux = 1
    flux_mode = flux_mode / np.max(np.abs(flux_mode))
    print(f"Elementary flux mode {i+1}:")
    for rxn, fl in zip(reactions, flux_mode):
        print(f"  {rxn:<15}: {fl:+.3f}")
    print()

In [ ]:
# Verify steady state: S @ v_combined should be ~0 for any v in null space
# Combine basis vectors with arbitrary coefficients
if ns.shape[1] > 0:
    coeffs = np.array([1.0] * ns.shape[1])
    v_steady = ns @ coeffs
    Sv = S_matrix @ v_steady
    print("Steady-state check (S·v should be ~0):")
    for met, sv_val in zip(metabolites, Sv):
        print(f"  d[{met}]/dt = {sv_val:.2e}")
    print()
    print("All values near zero — steady state confirmed.")

In [ ]:
# Visualise the flux distribution
if ns.shape[1] > 0:
    flux_display = ns[:, 0]
    flux_display = flux_display / np.max(np.abs(flux_display))

    colors_flux = ['steelblue' if f > 0 else 'tomato' for f in flux_display]

    fig, ax = plt.subplots(figsize=(7, 3))
    bars = ax.bar(reactions, flux_display, color=colors_flux)
    ax.axhline(0, color='k', lw=0.8)
    ax.set_ylabel('Relative Flux (normalised)')
    ax.set_title('Elementary Flux Mode 1 — Glycolysis Fragment')
    ax.set_xticklabels(reactions, rotation=20, ha='right')
    plt.tight_layout()
    plt.show()

---
## 8. Exercises

Apply your understanding of enzyme kinetics to the following problems.

### Exercise 1: Fit Michaelis-Menten to Experimental Data

The following data come from an assay of acetylcholinesterase. Fit the Michaelis-Menten model and report Km, Vmax, and their 95% CIs.

In [ ]:
# Acetylcholinesterase kinetics (realistic parameters)
# True Km ~ 0.09 mM, Vmax ~ 150 µmol/min/mg
ache_substrate_mM = np.array([0.01, 0.02, 0.05, 0.08, 0.10, 0.15, 0.20, 0.30, 0.50])
ache_velocity     = np.array([14.8, 26.3, 52.1, 72.4, 82.3, 101.7, 112.4, 126.8, 139.2])

# --- Your code here ---
# 1. Fit the MM equation to ache_substrate_mM and ache_velocity
# 2. Print Km, Vmax with 95% confidence intervals
# 3. Plot the data and fitted curve

p0_ex1 = [140, 0.08]
popt_ex1, pcov_ex1 = curve_fit(
    michaelis_menten, ache_substrate_mM, ache_velocity,
    p0=p0_ex1, bounds=([0, 0], [np.inf, np.inf])
)
perr_ex1 = np.sqrt(np.diag(pcov_ex1))

print(f"Vmax = {popt_ex1[0]:.2f} ± {1.96*perr_ex1[0]:.2f} µmol/min/mg")
print(f"Km   = {popt_ex1[1]:.4f} ± {1.96*perr_ex1[1]:.4f} mM")

s_ex1 = np.linspace(0, 0.55, 200)
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(ache_substrate_mM, ache_velocity, color='steelblue', s=60, zorder=5)
ax.plot(s_ex1, michaelis_menten(s_ex1, *popt_ex1), color='tomato', lw=2)
ax.set_xlabel('[S] (mM)')
ax.set_ylabel('v (µmol/min/mg)')
ax.set_title('Exercise 1: Acetylcholinesterase Kinetics')
plt.tight_layout()
plt.show()

### Exercise 2: Determine Inhibition Type from Kinetic Data

A researcher measured enzyme activity at two inhibitor concentrations. Determine whether inhibition is competitive, non-competitive, or uncompetitive.

In [ ]:
# Data: substrate in µM, velocity in µM/s
ex2_substrate = np.array([1, 2, 4, 8, 16, 32])

# Simulate non-competitive inhibition (hidden from student)
_ex2_vmax = 80.0; _ex2_km = 4.0; _ex2_ki = 5.0; _ex2_inh = 5.0
ex2_v_no_inh  = michaelis_menten(ex2_substrate, _ex2_vmax, _ex2_km)
ex2_v_with_inh = noncompetitive_inhibition(ex2_substrate, _ex2_vmax, _ex2_km, _ex2_inh, _ex2_ki)
ex2_v_no_inh  += rng.normal(0, 0.02 * ex2_v_no_inh)
ex2_v_with_inh += rng.normal(0, 0.02 * ex2_v_with_inh)

# --- Your analysis here ---
# Fit each inhibition model, compare apparent Km and Vmax,
# or examine the Lineweaver-Burk pattern.

# Step 1: Fit MM to each condition
p_no_inh,  _ = curve_fit(michaelis_menten, ex2_substrate, ex2_v_no_inh,  p0=[70,3], bounds=([0,0],[np.inf,np.inf]))
p_with_inh, _ = curve_fit(michaelis_menten, ex2_substrate, ex2_v_with_inh, p0=[50,4], bounds=([0,0],[np.inf,np.inf]))

print("Apparent kinetics:")
print(f"  No inhibitor:   Vmax = {p_no_inh[0]:.2f}, Km = {p_no_inh[1]:.2f}")
print(f"  With inhibitor: Vmax = {p_with_inh[0]:.2f}, Km = {p_with_inh[1]:.2f}")
print()

vmax_change_pct = 100 * (p_with_inh[0] - p_no_inh[0]) / p_no_inh[0]
km_change_pct   = 100 * (p_with_inh[1] - p_no_inh[1]) / p_no_inh[1]
print(f"  Vmax change: {vmax_change_pct:+.1f}%")
print(f"  Km change:   {km_change_pct:+.1f}%")
print()
if abs(km_change_pct) < 15 and vmax_change_pct < -10:
    print("Diagnosis: Km unchanged, Vmax decreased → Non-competitive inhibition")
elif km_change_pct > 15 and abs(vmax_change_pct) < 15:
    print("Diagnosis: Km increased, Vmax unchanged → Competitive inhibition")
else:
    print("Diagnosis: Both Km and Vmax changed → Uncompetitive or mixed inhibition")

### Exercise 3: Fit Hill Equation to Cooperative Binding Data

Aspartate transcarbamoylase (ATCase) shows cooperative kinetics. Fit the Hill equation to the data below and estimate n and K₀.₅.

In [ ]:
# ATCase-like cooperative kinetics: n ≈ 2.5, K0.5 ≈ 12 mM aspartate
atcase_substrate_mM = np.array([2, 4, 6, 8, 10, 12, 15, 20, 30, 50])
atcase_velocity     = np.array([3.1, 8.2, 15.4, 26.7, 40.1, 51.8, 67.3, 80.9, 91.4, 97.8])

# --- Your code here ---
# Fit the Hill equation: hill_equation(substrate, vmax, k_half, n)
# Report Vmax, K0.5, and Hill coefficient n

popt_ex3, pcov_ex3 = curve_fit(
    hill_equation, atcase_substrate_mM, atcase_velocity,
    p0=[100, 10, 2.0],
    bounds=([0, 0.1, 0.5], [200, 100, 8])
)
perr_ex3 = np.sqrt(np.diag(pcov_ex3))

print(f"Fitted Vmax = {popt_ex3[0]:.2f} ± {1.96*perr_ex3[0]:.2f}")
print(f"Fitted K0.5 = {popt_ex3[1]:.2f} ± {1.96*perr_ex3[1]:.2f} mM")
print(f"Hill n      = {popt_ex3[2]:.2f} ± {1.96*perr_ex3[2]:.2f}")
print()
if popt_ex3[2] > 1.2:
    print(f"n = {popt_ex3[2]:.2f} > 1: positive cooperativity (sigmoidal kinetics)")
elif popt_ex3[2] < 0.8:
    print(f"n = {popt_ex3[2]:.2f} < 1: negative cooperativity")
else:
    print(f"n = {popt_ex3[2]:.2f} ≈ 1: no cooperativity (Michaelis-Menten)")

s_ex3 = np.linspace(0, 55, 200)
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(atcase_substrate_mM, atcase_velocity, color='steelblue', s=60, zorder=5)
ax.plot(s_ex3, hill_equation(s_ex3, *popt_ex3), color='tomato', lw=2,
        label=f'Hill fit (n={popt_ex3[2]:.2f})')
ax.plot(s_ex3, michaelis_menten(s_ex3, popt_ex3[0], popt_ex3[1]),
        color='steelblue', lw=2, linestyle='--', label='MM (n=1)')
ax.set_xlabel('[Aspartate] (mM)')
ax.set_ylabel('v (nmol/min)')
ax.set_title('Exercise 3: ATCase Cooperativity')
ax.legend()
plt.tight_layout()
plt.show()

### Exercise 4: Build a Stoichiometric Matrix for a Metabolic Pathway

Construct the stoichiometric matrix for the TCA cycle fragment below and find the null space dimension.

```
R1: Acetyl-CoA + OAA → Citrate        (Citrate synthase)
R2: Citrate → Isocitrate              (Aconitase)
R3: Isocitrate → α-Ketoglutarate      (Isocitrate dehydrogenase)
R4: α-Ketoglutarate → Succinyl-CoA    (α-Ketoglutarate dehydrogenase)
R5: Succinyl-CoA → Succinate          (Succinyl-CoA synthetase)
R6: Succinate → Fumarate              (Succinate dehydrogenase)
R7: Fumarate → Malate                 (Fumarase)
R8: Malate → OAA                      (Malate dehydrogenase)
```

Metabolites: Acetyl-CoA (exchange), OAA, Citrate, Isocitrate, α-KG, Succinyl-CoA, Succinate, Fumarate, Malate

In [ ]:
# TCA cycle fragment stoichiometric matrix
# Metabolites (rows): OAA, Citrate, Isocitrate, aKG, Succinyl-CoA, Succinate, Fumarate, Malate
# (Acetyl-CoA treated as exchange metabolite — not tracked internally)
# Reactions (cols): R1 through R8

tca_metabolites = ['OAA', 'Citrate', 'Isocitrate', 'aKG', 'Succinyl-CoA', 'Succinate', 'Fumarate', 'Malate']
tca_reactions   = ['CS', 'Aconitase', 'IDH', 'aKGDH', 'SCS', 'SDH', 'Fumarase', 'MDH']

#              CS   Aco  IDH  aKGDH SCS  SDH  Fum  MDH
S_tca = np.array([
    [-1,   0,   0,   0,   0,   0,   0,   1 ],  # OAA
    [ 1,  -1,   0,   0,   0,   0,   0,   0 ],  # Citrate
    [ 0,   1,  -1,   0,   0,   0,   0,   0 ],  # Isocitrate
    [ 0,   0,   1,  -1,   0,   0,   0,   0 ],  # aKG
    [ 0,   0,   0,   1,  -1,   0,   0,   0 ],  # Succinyl-CoA
    [ 0,   0,   0,   0,   1,  -1,   0,   0 ],  # Succinate
    [ 0,   0,   0,   0,   0,   1,  -1,   0 ],  # Fumarate
    [ 0,   0,   0,   0,   0,   0,   1,  -1 ],  # Malate
], dtype=float)

ns_tca = null_space(S_tca)
rank_tca = np.linalg.matrix_rank(S_tca)

print(f"TCA fragment: {len(tca_metabolites)} metabolites × {len(tca_reactions)} reactions")
print(f"Rank of S = {rank_tca}")
print(f"Null space dimension = {ns_tca.shape[1]} (degrees of freedom)")
print()

if ns_tca.shape[1] > 0:
    mode = ns_tca[:, 0]
    mode = mode / np.max(np.abs(mode))
    print("Basis flux mode (normalised):")
    for rxn, fl in zip(tca_reactions, mode):
        print(f"  {rxn:<12}: {fl:+.3f}")
    print()
    Sv = S_tca @ mode
    print("Steady-state check (S·v):")
    for met, val in zip(tca_metabolites, Sv):
        print(f"  {met:<14}: {val:.2e}")

---
## Summary

| Concept | Key Equation / Tool | What It Gives You |
|---------|---------------------|-------------------|
| Beer-Lambert | A = ε·l·c | Convert absorbance to [concentration] |
| Michaelis-Menten | v = Vmax·[S]/(Km+[S]) | Km (affinity), Vmax (capacity) |
| NLS fitting | `scipy.optimize.curve_fit` | Unbiased parameter estimates with CIs |
| Lineweaver-Burk | 1/v vs 1/[S] | Visual but error-distorting |
| Eadie-Hofstee | v vs v/[S] | Better balanced errors than LB |
| Hanes-Woolf | [S]/v vs [S] | Most uniform error distribution |
| Competitive inhibition | α·Km, Vmax unchanged | Ki from Km_app vs [I] slope |
| Non-competitive | Km unchanged, Vmax/α | Ki from Vmax_app vs [I] slope |
| Uncompetitive | Both Km/α', Vmax/α' decrease | Parallel LB lines |
| Mixed inhibition | α·Km, α'·Vmax | Second-quadrant LB intersection |
| Hill equation | V = Vmax·[S]^n/(K₀.₅^n+[S]^n) | n > 1 = positive cooperativity |
| Hill plot | log(v/(Vmax-v)) vs log([S]) | Slope = n directly |
| EC numbering | EC n1.n2.n3.n4 | Standardised enzyme classification |
| Stoichiometric matrix | S·v = 0 at steady state | Null space = feasible flux modes |

**Key takeaways:**
1. Always use nonlinear least squares to fit MM — linearisation methods distort error structure
2. Inhibition type is diagnosed by changes in apparent Km and Vmax, confirmed by Lineweaver-Burk patterns
3. Hill n > 1 signals positive cooperativity; the sigmoidal curve provides a biological switch mechanism
4. Metabolic steady states live in the null space of the stoichiometric matrix

[← Previous: Modern Workflows](../../Tier_3_Applied_Bioinformatics/12_Modern_Workflows/01_modern_workflows.ipynb) | [Next: Genetic Engineering In Silico →](../../Tier_3_Applied_Bioinformatics/14_Genetic_Engineering_In_Silico/01_genetic_engineering_in_silico.ipynb)